# Phase 5 Cross-Seed β Sweep — K=1 Headline (Colab, read-only over snapshots)

**Active phase:** 5
**Purpose:** Cross-seed audit of β ∈ {3, 5, 10} at K=1 against the n=10 A1' substrates from [the headline experiment](https://github.com/Dypatterson/Neuro-AI/blob/main/scripts/colab_phase5_headline_n10.ipynb). Decided by GPT's pushback on [report 055](https://github.com/Dypatterson/Neuro-AI/blob/main/reports/055_phase5_headline_beta_sweep_seed17.md) and the K=1 correction in [report 056](https://github.com/Dypatterson/Neuro-AI/blob/main/reports/056_phase5_headline_beta_sweep_K1_seed17.md): seed 17 alone showed β=5 ~8× the β=10 magnitude at single-seed n=200 (ΔE_raw = +0.000103 vs +0.000013), still 50× below the 5.5e-3 magnitude floor.

**Question this notebook answers:**
- Does β=5 improve mean ΔE over β=10 *consistently* across all 10 seeds?
- Does the improvement move the noise-floor ratio meaningfully?
- Are the stronger report-053 seeds also stronger at β=5?
- Does `role < content < random` hold across seeds, or is ordering seed-fragile?
- Does step3 remain cancelling/uniform cross-seed (geometry audit drill-down)?

**Decision rule** (per the working-agreement discussion):
- β=5 *consistent and materially bigger* → keep β-axis open, then cue-regime aggregator.
- β=5 *inconsistent or still deeply sub-floor* → close β-axis, cue-regime aggregator next.
- β=5 *unexpectedly near floor* → rerun full n=10 headline at β=5 before any D-sweep.

**This is a drill-down, not a graduation experiment.** It does not retune any design-spec parameter (γ, K_main, ε, τ, formulation, content_distortion, binding_noise_std all fixed). No "winning cell" selection — the result classifies the β axis as open or closed.

Read-only over Drive snapshots. Writes one aggregate JSON + markdown summary back to Drive.


In [ ]:
# 1. Clone repo with the patched audit harness (commit 2a31a67+).
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git log --oneline -8

# Verify the harness + K=1 patch are present.
import subprocess, sys
markers = [
    ('--headline-beta-sweep',         'scripts/phase5_frozen_snapshot_audit.py',  'harness β-sweep mode'),
    ('energy_unbiased_step3_final',   'experiments/40_phase5_branching.py',       'step-3 telemetry'),
    ('class TestStep3ScoreBias',      'tests/test_phase5_branching.py',           'step-3 regression tests'),
]
for marker, fpath, label in markers:
    r = subprocess.run(['grep', '-n', marker, fpath], capture_output=True, text=True)
    status = 'OK' if r.returncode == 0 else 'MISSING'
    print(f'  [{status}] {label}: {marker} in {fpath}')
    if r.returncode != 0:
        sys.exit(1)


In [ ]:
# 2. Mount Drive.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RESULTS = '/content/drive/MyDrive/neuro-ai/results'
print('drive results root:', DRIVE_RESULTS)
print('contents:')
!ls {DRIVE_RESULTS} | grep phase5_headline | head -25


In [ ]:
# 3. Locate the 10 A1' snapshots from the headline run.
# Convention from scripts/colab_phase5_headline_n10.ipynb cell 13:
#   /content/drive/MyDrive/neuro-ai/results/phase5_headline_substrate_seed{N}/snapshots/phase3_phase4_w4_step1800.pt
import os
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 29]
DRIVE_RESULTS = '/content/drive/MyDrive/neuro-ai/results'

snapshot_paths = {}
missing = []
for s in SEEDS:
    p = f'{DRIVE_RESULTS}/phase5_headline_substrate_seed{s}/snapshots/phase3_phase4_w4_step1800.pt'
    if os.path.exists(p):
        size_mb = os.path.getsize(p) / (1024 * 1024)
        snapshot_paths[s] = p
        print(f'  seed {s:>3}: {p}  ({size_mb:.1f} MB)')
    else:
        missing.append(s)
        print(f'  seed {s:>3}: MISSING at {p}')

if missing:
    print()
    print(f'!!! {len(missing)} snapshots missing: {missing}')
    print('Either run the headline notebook to produce them, or update the path convention above.')


In [ ]:
# 4. Parent-CPU sanity — do NOT touch CUDA in the parent (CLAUDE.md memory:
#    colab_workflow.md — parent must not init CUDA before launching workers).
#    Here we run sequentially in-process; the harness handles CUDA via --device cuda.
import sys, os
sys.path.insert(0, 'src')

# Light parent-side import check (CPU-only).
import importlib.util
spec = importlib.util.spec_from_file_location(
    'exp40', 'experiments/40_phase5_branching.py'
)
m = importlib.util.module_from_spec(spec)
sys.modules['exp40'] = m
spec.loader.exec_module(m)
print('experiments/40 loaded; settle_branch_with_prior has score_bias:',
      'score_bias' in __import__('inspect').signature(m.settle_branch_with_prior).parameters)

# GPU info but DO NOT initialize via torch.
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv | head -3


In [ ]:
# 5. Sequential cross-seed β sweep.
# K=1 (matches report 053 + 056), n_cues=200 (matches report 056), β ∈ {3, 5, 10}.
# Each seed takes ~1-3 min on a T4/A100; 10 seeds ~20-30 min total.

import subprocess, time, os, json
from pathlib import Path

BETAS = '3,5,10'
N_CUES = 200
K_MAIN = 1
GAMMA = 0.5
RUN_TAG = 'phase5_cross_seed_beta_sweep'
out_root = Path(f'reports/{RUN_TAG}')
out_root.mkdir(parents=True, exist_ok=True)

env = {**os.environ, 'PYTHONPATH': '/content/Neuro-AI/src'}

per_seed_jsons = {}
total_start = time.time()
for seed, snapshot in snapshot_paths.items():
    out_path = out_root / f'seed{seed}.json'
    print(f'[seed {seed}] running β sweep on {snapshot}', flush=True)
    t0 = time.time()
    r = subprocess.run([
        'python', 'scripts/phase5_frozen_snapshot_audit.py',
        '--snapshot', snapshot,
        '--output', str(out_path),
        '--headline-beta-sweep',
        '--headline-k-main', str(K_MAIN),
        '--headline-n-cues', str(N_CUES),
        '--headline-gamma', str(GAMMA),
        '--betas', BETAS,
        '--device', 'cuda',
    ], env=env, capture_output=True, text=True)
    dt = time.time() - t0
    if r.returncode != 0:
        print(f'  !!! FAILED in {dt:.1f}s')
        print(r.stdout[-1000:])
        print(r.stderr[-1500:])
        continue
    per_seed_jsons[seed] = out_path
    # Quick per-β preview.
    with open(out_path) as f:
        d = json.load(f)
    sw = d['headline_beta_sweep']
    summary = []
    for key, s in sw['by_beta'].items():
        b = key.replace('beta_', '')
        summary.append(f"β={b}: ΔE={s['delta_e_raw_mean']:+.5f}")
    print(f'  done in {dt:.1f}s — {"  ".join(summary)}', flush=True)

print(f'\n[total] {time.time() - total_start:.1f}s across {len(per_seed_jsons)} seeds')


In [ ]:
# 6. Aggregate cross-seed.
import json, math, statistics
from pathlib import Path

agg = {
    'config': {
        'seeds': sorted(per_seed_jsons),
        'betas': [3.0, 5.0, 10.0],
        'k_main': K_MAIN,
        'n_cues_per_seed': N_CUES,
        'gamma': GAMMA,
    },
    'per_seed': {},
    'cross_seed': {},
}

# Load per-seed results.
for seed, path in per_seed_jsons.items():
    with open(path) as f:
        d = json.load(f)
    sw = d['headline_beta_sweep']
    geom = d['geometry']
    agg['per_seed'][seed] = {
        'snapshot': sw['snapshot'],
        'n_atoms': sw['n_atoms'],
        'coverage_lambda': sw['coverage_lambda'],
        'bias_cv': geom.get('bias_cv'),
        'step3_shift_invariant_likely': geom.get('step3_shift_invariant_likely'),
        'effective_strength': geom.get('effective_strength'),
        'by_beta': {
            k: {
                'delta_e_raw_mean': v['delta_e_raw_mean'],
                'delta_e_step3_mean': v['delta_e_step3_mean'],
                'delta_e_raw_frac_positive': v['delta_e_raw_frac_positive'],
                'delta_e_step3_frac_positive': v['delta_e_step3_frac_positive'],
                'per_condition_mean_e_min_raw': v['per_condition_mean_e_min_raw'],
                'energy_ordering_low_to_high_raw': v['energy_ordering_low_to_high_raw'],
            } for k, v in sw['by_beta'].items()
        }
    }

# Cross-seed stats per β.
for beta_key in ('beta_3', 'beta_5', 'beta_10'):
    dE_raw    = [agg['per_seed'][s]['by_beta'][beta_key]['delta_e_raw_mean']    for s in sorted(per_seed_jsons)]
    dE_step3  = [agg['per_seed'][s]['by_beta'][beta_key]['delta_e_step3_mean']  for s in sorted(per_seed_jsons)]
    f_pos_raw = [agg['per_seed'][s]['by_beta'][beta_key]['delta_e_raw_frac_positive'] for s in sorted(per_seed_jsons)]
    orderings = [tuple(agg['per_seed'][s]['by_beta'][beta_key]['energy_ordering_low_to_high_raw']) for s in sorted(per_seed_jsons)]

    n = len(dE_raw)
    mean_raw   = statistics.mean(dE_raw)
    std_raw    = statistics.stdev(dE_raw) if n > 1 else 0.0
    mean_step3 = statistics.mean(dE_step3)
    std_step3  = statistics.stdev(dE_step3) if n > 1 else 0.0
    seeds_positive = sum(1 for x in dE_raw if x > 0)
    n_role_lt_content_lt_random = sum(1 for o in orderings if o == ('role', 'content', 'random'))

    # 95% CI on the mean (t-approximation with t_0.975 ≈ 2.262 for df=9).
    se_raw = std_raw / math.sqrt(n) if n > 0 else 0.0
    ci_raw = (mean_raw - 2.262 * se_raw, mean_raw + 2.262 * se_raw)

    agg['cross_seed'][beta_key] = {
        'n_seeds': n,
        'mean_delta_e_raw': mean_raw,
        'std_delta_e_raw_across_seeds': std_raw,
        'ci95_delta_e_raw': ci_raw,
        'noise_floor_ratio_raw': mean_raw / 5.5e-3,
        'seeds_positive_raw': seeds_positive,
        'mean_delta_e_step3': mean_step3,
        'std_delta_e_step3_across_seeds': std_step3,
        'n_orderings_role_lt_content_lt_random': n_role_lt_content_lt_random,
        'per_seed_dE_raw':   dict(zip(sorted(per_seed_jsons), dE_raw)),
        'per_seed_dE_step3': dict(zip(sorted(per_seed_jsons), dE_step3)),
        'mean_frac_positive_raw': statistics.mean(f_pos_raw),
    }

# Write aggregate JSON.
agg_path = out_root / 'cross_seed_aggregate.json'
with open(agg_path, 'w') as f:
    json.dump(agg, f, indent=2)
print('wrote', agg_path)


In [ ]:
# 7. Print summary table + decision flag.
print(f'{"β":>4}  {"n":>3}  {"mean_ΔE_raw":>12}  {"95% CI":>30}  {"std_seed":>9}  {"seeds>0":>8}  {"role<cont<rand":>14}  {"ΔE/floor":>9}')
for beta_key in ('beta_3', 'beta_5', 'beta_10'):
    s = agg['cross_seed'][beta_key]
    beta = beta_key.replace('beta_', '')
    ci_str = f'[{s["ci95_delta_e_raw"][0]:+.5f}, {s["ci95_delta_e_raw"][1]:+.5f}]'
    print(f'{beta:>4}  {s["n_seeds"]:>3}  {s["mean_delta_e_raw"]:>+12.6f}  {ci_str:>30}  '
          f'{s["std_delta_e_raw_across_seeds"]:>9.5f}  {s["seeds_positive_raw"]:>2}/{s["n_seeds"]:<4}  '
          f'{s["n_orderings_role_lt_content_lt_random"]:>3}/{s["n_seeds"]:<8}  {s["noise_floor_ratio_raw"]:>9.4f}')

# Decision banner.
print()
b5  = agg['cross_seed']['beta_5']
b10 = agg['cross_seed']['beta_10']
ratio_improvement = (abs(b5['mean_delta_e_raw']) / abs(b10['mean_delta_e_raw'])
                     if abs(b10['mean_delta_e_raw']) > 1e-12 else float('inf'))
b5_floor = b5['noise_floor_ratio_raw']
print(f'β=5 vs β=10: magnitude ratio = {ratio_improvement:.2f}x')
print(f'β=5 distance to magnitude floor: {b5_floor:.4f} (need >= 1.0 to graduate)')
print()
if b5['seeds_positive_raw'] >= 8 and abs(b5['mean_delta_e_raw']) > 2 * abs(b10['mean_delta_e_raw']):
    if b5_floor >= 0.5:
        print('VERDICT: β=5 consistent AND materially closer to floor — KEEP β-AXIS OPEN; rerun full n=10 headline at β=5 next.')
    else:
        print('VERDICT: β=5 consistent but still deep sub-floor — CLOSE β-AXIS; cue-regime aggregator next.')
elif b5['seeds_positive_raw'] < 6:
    print('VERDICT: β=5 inconsistent (cross-seed disagreement) — CLOSE β-AXIS; cue-regime aggregator next.')
else:
    print('VERDICT: β=5 ambiguous improvement — present to user for call.')


In [ ]:
# 8. Copy aggregate + per-seed JSONs back to Drive.
import shutil, os
DRIVE_RESULTS = '/content/drive/MyDrive/neuro-ai/results'
dst = f'{DRIVE_RESULTS}/{RUN_TAG}'
os.makedirs(dst, exist_ok=True)
src = f'reports/{RUN_TAG}'
shutil.copytree(src, dst, dirs_exist_ok=True)
print(f'copied {src} → {dst}')
!ls -la {dst} | head -20
